In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bsthere/nike-global-catalogue-2026")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'nike-global-catalogue-2026' dataset.
Path to dataset files: /kaggle/input/nike-global-catalogue-2026


In [ ]:
import pandas as pd
import os

# Assuming 'Global_Nike.csv' contains the consolidated data
global_nike_file = os.path.join(path, 'Global_Nike.csv')

if os.path.exists(global_nike_file):
    try:
        df = pd.read_csv(global_nike_file)
        print(f"Successfully loaded Global_Nike.csv. Total rows: {df.shape[0]}")
    except Exception as e:
        print(f"Error loading Global_Nike.csv: {e}")
else:
    print("Global_Nike.csv not found in the dataset directory. Please check the dataset path.")


KeyboardInterrupt: 

### Step 1: Structural Check & Memory Optimization
First, let's see how much data we are dealing with and check for missing values.

In [ ]:
# Check the shape of the data
print(f"Dataset Scale: {df.shape[0]:,} rows and {df.shape[1]} columns.\n")

# Check data types and null values
print("--- Data Types and Missing Values ---")
print(df.info())

# Inspect the top missing value columns
missing_data = df.isnull().sum().sort_values(ascending=False)
print("\n--- Missing Value Count Per Column ---")
print(missing_data[missing_data > 0])


Dataset Scale: 1,447,795 rows and 35 columns.

--- Data Types and Missing Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447795 entries, 0 to 1447794
Data columns (total 35 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   snapshot_date          1447795 non-null  object 
 1   country_code           1447795 non-null  object 
 2   product_name           1447795 non-null  object 
 3   model_number           1447795 non-null  object 
 4   currency               1447795 non-null  object 
 5   price_local            1447795 non-null  float64
 6   sale_price_local       210687 non-null   float64
 7   gender_segment         1441048 non-null  object 
 8   size_label             1447793 non-null  object 
 9   category               1447795 non-null  object 
 10  subcategory            1447767 non-null  object 
 11  product_id             1447795 non-null  object 
 12  sku                    1447793 non-null  

### Step 2: Clean and Convert Pricing Columns

In [ ]:
# Fill missing `sale_price_local` values with `price_local` values
df['sale_price_local'] = df['sale_price_local'].fillna(df['price_local'])

print("--- Updated Missing Values for Sale Price Local ---")
print(df['sale_price_local'].isnull().sum())

display(df[['price_local', 'sale_price_local']].head())

--- Updated Missing Values for Sale Price Local ---
0


,price_local,sale_price_local
0,110.0,110.0
1,110.0,110.0
2,110.0,110.0
3,110.0,110.0
4,110.0,110.0


### Step 3: Outlier & Anomaly Detection

Let's check for common anomalies like items priced at $0.00 or sale prices being higher than the original price.

In [ ]:
# Anomaly 1: Items priced at $0.00
zero_price_anomalies = df[(df['price_local'] == 0) | (df['sale_price_local'] == 0)]
print(f"Number of items with a local price of $0.00: {len(zero_price_anomalies)}")
if not zero_price_anomalies.empty:
    display(zero_price_anomalies.head())

Number of items with a local price of $0.00: 18


,snapshot_date,country_code,product_name,model_number,currency,price_local,sale_price_local,gender_segment,size_label,category,...,canonical_url,image_url,gtin,stock_keeping_unit_id,catalog_sku_id,nike_size,localized_size,size_conversion_id,sport_tags,record_source
37397,2026-03-19,US,Nike Digital Gift Card,GIFTCARD,USD,0.0,0.0,MEN|WOMEN,EN,DIGITAL_GIFT_CARD,...,https://www.nike.com/t/digital-gift-card-email...,https://secure-images.nike.com/is/image/DotCom...,NaN,1.003963e+09,ca2d70f0-68ad-4ec4-9c1d-f12284f1b3a8,EN,EN,NaN,Tennis|Lifestyle|Cheerleading|Soccer|Golf|Runn...,thread_exact
89222,2026-03-19,FI,Nike Gift Card,GIFTCARD,EUR,0.0,0.0,MEN|WOMEN,EN,PHYSICAL_GIFT_CARD,...,https://www.nike.com/fi/t/gift-card-2F6OXBmk/G...,https://secure-images.nike.com/is/image/DotCom...,NaN,2.711961e+07,52a70174-5383-3521-ac6b-e2d2e678cad0,EN,EN,NaN,Lifestyle,thread_exact
129386,2026-03-19,JP,ナイキ ギフトカード（デジタル形式）,GIFTCARD,JPY,0.0,0.0,MEN|BOYS|WOMEN|GIRLS,JP,DIGITAL_GIFT_CARD,...,https://www.nike.com/jp/t/ナイキ-ギフトカード（デジタル形式）を約...,https://secure-images.nike.com/is/image/DotCom...,NaN,2.547256e+07,8c7a5f45-a339-3d81-920b-7ce190b8c047,JP,JP,NaN,Lifestyle,thread_exact
172515,2026-03-19,SI,Nike Gift Card,GIFTCARD,EUR,0.0,0.0,MEN|WOMEN,EN,PHYSICAL_GIFT_CARD,...,https://www.nike.com/si/t/gift-card-2F6OXBmk/G...,https://secure-images.nike.com/is/image/DotCom...,NaN,2.711961e+07,52a70174-5383-3521-ac6b-e2d2e678cad0,EN,EN,NaN,Lifestyle,thread_exact
215803,2026-03-19,IE,Nike Gift Card,GIFTCARD,EUR,0.0,0.0,MEN|WOMEN,EN,PHYSICAL_GIFT_CARD,...,https://www.nike.com/ie/t/gift-card-2F6OXBmk/G...,https://secure-images.nike.com/is/image/DotCom...,NaN,2.711961e+07,52a70174-5383-3521-ac6b-e2d2e678cad0,EN,EN,NaN,Lifestyle,thread_exact


In [ ]:
# Anomaly 2: Sale price higher than original price
higher_sale_price_anomalies = df[df['sale_price_local'] > df['price_local']]
print(f"Number of items where sale price is higher than original price: {len(higher_sale_price_anomalies)}")
if not higher_sale_price_anomalies.empty:
    display(higher_sale_price_anomalies.head())

Number of items where sale price is higher than original price: 210687


,snapshot_date,country_code,product_name,model_number,currency,price_local,sale_price_local,gender_segment,size_label,category,...,canonical_url,image_url,gtin,stock_keeping_unit_id,catalog_sku_id,nike_size,localized_size,size_conversion_id,sport_tags,record_source
50,2026-03-19,US,A' One,IF2283,USD,21.97,30.0,BOYS|GIRLS,L,APPAREL,...,https://www.nike.com/t/a-one-big-kids-dri-fit-...,https://secure-images.nike.com/is/image/DotCom...,NaN,1.005269e+09,fd08533a-efd8-3c9a-9d02-5943f036bfd4,L,L,54e8515a-3e2a-3186-80c4-645e58c1eac9,Workouts|Training & Gym,thread_exact
51,2026-03-19,US,A' One,IF2283,USD,21.97,30.0,BOYS|GIRLS,M,APPAREL,...,https://www.nike.com/t/a-one-big-kids-dri-fit-...,https://secure-images.nike.com/is/image/DotCom...,NaN,1.005269e+09,3d98052c-166b-3598-9827-22a1416b31df,M,M,bb540851-e674-3537-b539-8ca837ee030a,Workouts|Training & Gym,thread_exact
52,2026-03-19,US,A' One,IF2283,USD,21.97,30.0,BOYS|GIRLS,S,APPAREL,...,https://www.nike.com/t/a-one-big-kids-dri-fit-...,https://secure-images.nike.com/is/image/DotCom...,NaN,1.005269e+09,359121b5-edd3-3254-83cf-62b9617ed068,S,S,c53217e0-f6a5-3991-a216-ae07141ea17d,Workouts|Training & Gym,thread_exact
53,2026-03-19,US,A' One,IF2283,USD,21.97,30.0,BOYS|GIRLS,XL,APPAREL,...,https://www.nike.com/t/a-one-big-kids-dri-fit-...,https://secure-images.nike.com/is/image/DotCom...,NaN,1.005269e+09,80bc7ec8-c0ba-3cb6-9c44-0aa766647874,XL,XL,b8edc906-4a82-3fcd-8033-3846113964df,Workouts|Training & Gym,thread_exact
54,2026-03-19,US,A' One,IF2283,USD,21.97,30.0,BOYS|GIRLS,XS,APPAREL,...,https://www.nike.com/t/a-one-big-kids-dri-fit-...,https://secure-images.nike.com/is/image/DotCom...,NaN,1.005269e+09,db277755-a6e3-3f4e-8cc7-c1b03e0ea585,XS,XS,0495926f-1505-32f3-985c-e6eaf5868f94,Workouts|Training & Gym,thread_exact


### Handling Anomalies

In [ ]:
# For items where sale price is higher than original price, set sale_price_local to price_local
df.loc[df['sale_price_local'] > df['price_local'], 'sale_price_local'] = df['price_local']

# Verify the correction
higher_sale_price_anomalies_after_correction = df[df['sale_price_local'] > df['price_local']]
print(f"Number of items where sale price is higher than original price after correction: {len(higher_sale_price_anomalies_after_correction)}")
display(df[['price_local', 'sale_price_local']].head())

Number of items where sale price is higher than original price after correction: 0


,price_local,sale_price_local
0,110.0,110.0
1,110.0,110.0
2,110.0,110.0
3,110.0,110.0
4,110.0,110.0


### Step 4: Categorical Uniformity (The String Clean Up)

Let's make sure text categorical labels are standardized across all 45 country markets.

In [ ]:
# Standardize text constraints to upper case to remove case-sensitivity discrepancies
if 'gender_segment' in df.columns:
    # Fill NaN values with a placeholder like 'UNKNOWN', then replace string 'nan' variations
    df['gender_segment'] = df['gender_segment'].fillna('UNKNOWN').astype(str).str.replace('nan', 'UNKNOWN', case=False, regex=True).str.upper().str.strip()
if 'category' in df.columns:
    df['category'] = df['category'].astype(str).str.upper().str.strip()

# Check unique values for core categorical columns after standardization
categorical_cols = ['category', 'gender_segment', 'currency']

for col in categorical_cols:
    if col in df.columns:
        print(f"\n--- Unique Categories in '{col}' ({df[col].nunique()} values) ---")
        print(df[col].value_counts().head(10))


--- Unique Categories in 'category' (5 values) ---
category
FOOTWEAR              724811
APPAREL               696577
EQUIPMENT              26389
PHYSICAL_GIFT_CARD        16
DIGITAL_GIFT_CARD          2
Name: count, dtype: int64

--- Unique Categories in 'gender_segment' (14 values) ---
gender_segment
MEN                     579987
WOMEN                   373417
MEN|WOMEN               231852
BOYS|GIRLS              192882
GIRLS                    32659
BOYS                     27176
UNKNOWN                   6747
MEN|BOYS|WOMEN|GIRLS      2629
WOMEN|GIRLS                253
BOYS|WOMEN|GIRLS           110
Name: count, dtype: int64

--- Unique Categories in 'currency' (29 values) ---
currency
EUR    719260
USD     69949
JPY     43306
GBP     39765
DKK     39362
PLN     39186
SEK     38519
CHF     38088
CNY     37547
ILS     36874
Name: count, dtype: int64


### Step 5: Pricing Standardization & Key Metric Creation
To enable cross-country comparisons and analyze pricing strategies, we'll standardize all local prices to USD. Additionally, we'll calculate key metrics like markdown depth, cross-border premium, and inventory availability signals.

In [ ]:
# Expanded Global Exchange Rate Map to 1 USD (Updated for 2026 baseline metrics)
usd_rates = {
    # Americas
    'USD': 1.00,   # United States
    'CAD': 0.71,   # Canada
    'MXN': 0.054,  # Mexico
    'BRL': 0.18,   # Brazil
    'ARS': 0.0011, # Argentina
    'CLP': 0.0011, # Chile

    # Europe (Eurozone & Non-Euro)
    'EUR': 1.08,   # Eurozone (France, Germany, Italy, Spain, etc.)
    'GBP': 1.26,   # United Kingdom
    'CHF': 1.12,   # Switzerland
    'SEK': 0.093,  # Sweden
    'NOK': 0.091,  # Norway
    'DKK': 0.14,   # Denmark
    'PLN': 0.25,   # Poland
    'CZK': 0.042,  # Czech Republic
    'HUF': 0.0027, # Hungary
    'TRY': 0.029,  # Turkey

    # Asia & Middle East
    'INR': 0.0115, # India
    'JPY': 0.0063, # Japan
    'CNY': 0.14,   # China
    'KRW': 0.00072,# South Korea
    'SGD': 0.74,   # Singapore
    'HKD': 0.13,   # Hong Kong
    'TWD': 0.031,  # Taiwan
    'MYR': 0.21,   # Malaysia
    'THB': 0.028,  # Thailand
    'IDR': 0.000062,# Indonesia
    'PHP': 0.017,  # Philippines
    'VND': 0.000039,# Vietnam
    'AED': 0.27,   # United Arab Emirates
    'SAR': 0.27,   # Saudi Arabia
    'ILS': 0.27,   # Israel

    # Oceania & Africa
    'AUD': 0.65,   # Australia
    'NZD': 0.59,   # New Zealand
    'ZAR': 0.053   # South Africa
}

In [ ]:
import numpy as np

# 1. Broad Global Exchange Rate Map to USD
# Use the comprehensive usd_rates dictionary defined above
df['exchange_rate'] = df['currency'].map(usd_rates).fillna(1.0)

# Calculate unified standardized pricing
df['price_usd'] = df['price_local'] * df['exchange_rate']
df['sale_price_usd'] = df['sale_price_local'] * df['exchange_rate']

# 2. Metric A: Calculate Markdown Depth for every listing
df['discount_depth_pct'] = (df['price_local'] - df['sale_price_local']) / df['price_local']
df['is_discounted'] = df['discount_depth_pct'] > 0

# 3. Metric B: Cross-Border Premium Setup (Isolate US Base Prices)
# Create a reference mapping of unique model numbers and their baseline US retail prices
us_prices = df[df['country_code'].str.upper() == 'US'].groupby('model_number')['price_usd'].median().to_dict()
df['us_baseline_price_usd'] = df['model_number'].map(us_prices)

# Calculate Premium percentage vs US baseline (Ignore if product doesn't exist in US catalog)
df['cross_border_premium_pct'] = np.where(
    df['us_baseline_price_usd'].notnull(),
    ((df['price_usd'] - df['us_baseline_price_usd']) / df['us_baseline_price_usd']) * 100,
    np.nan
)

# 4. Metric C: Extract Inventory Availability Signals
# Ensure 'available' column is clean boolean
if df['available'].dtype == 'object':
    df['available'] = df['available'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})
df['is_stockout'] = df['available'] == False

print("Business logic applied successfully. Preparing regional aggregations...")


Business logic applied successfully. Preparing regional aggregations...


In [ ]:
print('Unique values in country_code column:')
print(df['country_code'].unique())
print(f"Number of entries with 'US' in country_code: {df[df['country_code'].str.upper() == 'US'].shape[0]}")

Unique values in country_code column:
['US' 'FI' 'JP' 'SI' 'IE' 'GB' 'LU' 'AT' 'BE' 'FR' 'IT' 'NL' 'PL' 'DK'
 'ES' 'GR' 'HU' 'SE' 'CZ' 'DE' 'PT' 'CN' 'RO' 'SK' 'HR' 'IL' 'ZA' 'CH'
 'BG' 'TR' 'CA' 'MX' 'PH' 'SG' 'TH' 'MY' 'TW' 'ID' 'VN' 'KR' 'NO' 'NZ'
 'IN' 'AU' 'EG']
Number of entries with 'US' in country_code: 69949


### Step 6: Aggregate & Compress Into a Lightweight Tableau File
To create a performant Tableau dashboard, we'll aggregate the detailed transactional data into a more compact Country-by-Product matrix summary. This will significantly reduce file size while retaining crucial insights for analysis.

In [ ]:
# Define aggregation dictionary
agg_funcs = {
    'price_usd': 'median',
    'sale_price_usd': 'median',
    'discount_depth_pct': 'mean',
    'is_discounted': 'sum',
    'cross_border_premium_pct': 'mean',
    'is_stockout': 'sum',
    'product_id': 'nunique' # Count unique product IDs per group
}

# Group by relevant dimensions
df_agg = df.groupby(['country_code', 'category', 'gender_segment', 'model_number', 'product_name', 'currency']).agg(**{
    'median_price_usd': ('price_usd', 'median'),
    'median_sale_price_usd': ('sale_price_usd', 'median'),
    'avg_discount_depth_pct': ('discount_depth_pct', 'mean'),
    'num_discounted_listings': ('is_discounted', 'sum'),
    'avg_cross_border_premium_pct': ('cross_border_premium_pct', 'mean'),
    'num_stockout_listings': ('is_stockout', 'sum'),
    'unique_products_in_group': ('product_id', 'nunique')
}).reset_index()

# Convert boolean sums to percentages if desired (optional, can also be done in Tableau)
df_agg['pct_discounted_listings'] = (df_agg['num_discounted_listings'] / df_agg['unique_products_in_group']) * 100


# Display the aggregated data
print(f"Aggregated DataFrame shape: {df_agg.shape[0]:,} rows and {df_agg.shape[1]} columns.")
display(df_agg.head())

# Save the aggregated DataFrame to a CSV file in a writable directory
# Changing output path from 'path' (read-only) to current working directory
output_file_path = 'nike_global_final.csv'
df_agg.to_csv(output_file_path, index=False)
print(f"Aggregated data saved to: {os.path.abspath(output_file_path)}")

Aggregated DataFrame shape: 171,308 rows and 14 columns.


,country_code,category,gender_segment,model_number,product_name,currency,median_price_usd,median_sale_price_usd,avg_discount_depth_pct,num_discounted_listings,avg_cross_border_premium_pct,num_stockout_listings,unique_products_in_group,pct_discounted_listings
0,AT,APPAREL,BOYS,DA1806,Nike Sportswear Air Max,EUR,70.1892,70.1892,0.0,0,NaN,2,1,0.0
1,AT,APPAREL,BOYS,DD3055,Nike Dri-FIT Miler,EUR,30.2292,30.2292,0.0,0,NaN,0,1,0.0
2,AT,APPAREL,BOYS,DH0389,Nike Sportswear Club Fleece,EUR,43.1892,43.1892,0.0,0,NaN,0,1,0.0
3,AT,APPAREL,BOYS,DN1970,Nike Dri-FIT Victory,EUR,32.3892,32.3892,0.0,0,NaN,1,1,0.0
4,AT,APPAREL,BOYS,DO1968,Jordan,EUR,53.9892,53.9892,0.0,0,NaN,1,1,0.0


Aggregated data saved to: /content/nike_global_summary_tableau.csv
